In [1]:
!pip install seaborn
import warnings
warnings.filterwarnings("ignore", category=Warning)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
df = pd.read_csv("data/preprocessed_songs.csv")
genre_cols = df.columns[7:]
df = df[df[genre_cols].sum(axis=1) > 0]
df = df.reset_index(drop=True)
df.head()

,Unnamed: 0,artist,track,listeners,playcount,tags,plct/lstn,Disco,Grunge,Hip-Hop,...,pop rock,punk,rap,rnb,rock,shoegaze,soft rock,soul,synthpop,trap
0,0,Harry Styles,Aperture,524766,4295814,"['House', 'electronic', 'alternative', 'Disco']",8.186152,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,Addison Rae,Fame Is a Gun,959674,17809028,"['electropop', 'pop', 'synthpop', 'dance-pop']",18.557373,0,0,0,...,0,0,0,0,0,0,0,0,1,0
2,3,Zara Larsson,Midnight Sun,620819,10022037,['pop'],16.143251,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,Dominic Fike,Babydoll,1425059,21116521,"['alternative', 'indie', 'pop']",14.817998,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Djo,End of Beginning,1946092,32806375,"['indie', 'pop', 'indie pop', 'rock']",16.857566,0,0,0,...,0,0,0,0,1,0,0,0,0,0


In [3]:
dfclean = df.drop(columns=["Unnamed: 0", "artist", "track", "tags"])
dfclean.head()

,listeners,playcount,plct/lstn,Disco,Grunge,Hip-Hop,House,Korean,Kpop,Love,...,pop rock,punk,rap,rnb,rock,shoegaze,soft rock,soul,synthpop,trap
0,524766,4295814,8.186152,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,959674,17809028,18.557373,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
2,620819,10022037,16.143251,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1425059,21116521,14.817998,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1946092,32806375,16.857566,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


In [4]:
#Standardization
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(dfclean)


In [5]:
from sklearn.neighbors import NearestNeighbors
model = NearestNeighbors(n_neighbors=10, metric='cosine')
model.fit(X_scaled)


,n_neighbors,10
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [6]:
#Test out the model here with one song 
song_index = 0  

distances, indices = model.kneighbors(X_scaled[song_index].reshape(1, -1))

recommended_songs = df.iloc[indices[0]]
print(recommended_songs[["artist", "track"]])

#Test similarities of active genres from our song and reccs
original = df.iloc[song_index][genre_cols]
recs = df.iloc[indices[0]][genre_cols]
active_genres = original[original == 1].index
if len(active_genres) == 0:
    consistency_score = 0
else:
    matches = recs[active_genres].sum(axis=1)
    consistency_score = matches.mean() / len(active_genres)

print("Similarity Score:", consistency_score)

             artist                                              track
0      Harry Styles                                           Aperture
2333      Daft Punk  Get Lucky (feat. Pharrell Williams and Nile Ro...
3233      Daft Punk                                 Giorgio by Moroder
1654      Bad Bunny                                          El Apagón
3910       Dua Lipa                                        Hallucinate
4800     KAYTRANADA                    Witchy (feat. Childish Gambino)
5281  Martin Garrix                                Scared to Be Lonely
4583   Fred again..                                       leavemealone
3312  Calvin Harris                                             My Way
5493           Alok                                        Hear Me Now
Similarity Score: 0.6


In [16]:
#Full functions for every single song 
def recommend (track_name,k=10):
    index = get_index(track_name)
    if index is None:
        return None, 0
    recommended_songs, indices = get_reccs(index)
    score = get_similarity_score(index,indices)
    return score
def get_index(track_name):
    matches = df[(df["track"].str.lower() == track_name.lower())]
    if matches.empty:
        print("Track name not found. Try again.")
        return None
    return matches.index[0]

def get_reccs(song_index):
    distances, indices = model.kneighbors(X_scaled[song_index].reshape(1, -1))
    indices = indices[0][1:]
    recommended_songs = df.iloc[indices]
    print(recommended_songs[["artist", "track"]])
    return recommended_songs, indices

#Overall overlap
def get_similarity_score(song_index,indices):
    original = df.iloc[song_index][genre_cols]
    recs = df.iloc[indices][genre_cols]
    
    active_genres = original[original == 1].index
    matches = recs[active_genres].sum(axis=1)
    consistency_score = matches.mean() / len(active_genres)
    
    print("Similarity Score:", consistency_score)
    return consistency_score

In [17]:
recommend("Aperture")

             artist                                              track
2333      Daft Punk  Get Lucky (feat. Pharrell Williams and Nile Ro...
3233      Daft Punk                                 Giorgio by Moroder
1654      Bad Bunny                                          El Apagón
3910       Dua Lipa                                        Hallucinate
4800     KAYTRANADA                    Witchy (feat. Childish Gambino)
5281  Martin Garrix                                Scared to Be Lonely
4583   Fred again..                                       leavemealone
3312  Calvin Harris                                             My Way
5493           Alok                                        Hear Me Now
Similarity Score: 0.5555555555555556


np.float64(0.5555555555555556)

In [9]:
recommend("Fame is a gun")

                artist                            track
3747      Taylor Swift                 I Think He Knows
402         Charli xcx                            Apple
3828  Carly Rae Jepsen                 Run Away with Me
95       Chappell Roan               Red Wine Supernova
2097          Dua Lipa                         Physical
5439              Jade               Angel Of My Dreams
4166             TWICE                            LIKEY
1097      Taylor Swift  I Can Do It with a Broken Heart
896          Lady Gaga                          Disease
Similarity Score: 0.9444444444444444


np.float64(0.9444444444444444)

In [18]:
recommend("Babydoll")

                artist                        track
1413     Billie Eilish         lovely (with Khalid)
249       Lana Del Rey                   West Coast
3530      Lana Del Rey                       Lolita
3588      Lana Del Rey                    Old Money
3595      Lana Del Rey            Money Power Glory
2716  Melanie Martinez                    Dollhouse
3500      Lana Del Rey  Fucked My Way Up to the Top
803         Tate McRae            run for the hills
3782    Regina Spektor                    Two Birds
Similarity Score: 1.0


np.float64(1.0)

In [11]:
recommend("End of Beginning")

             artist                                              track
345          Cafuné                                             Tek It
235          Mitski                              Washing Machine Heart
41       The Smiths  There Is a Light That Never Goes Out - 2011 Re...
535    Lana Del Rey                                        Born to Die
462    Lana Del Rey                                        Video Games
368    Lana Del Rey                                 Summertime Sadness
537    Lana Del Rey                                  Diet Mountain Dew
1516   Lana Del Rey                                    National Anthem
1141  Billie Eilish                                everything i wanted
Similarity Score: 0.7777777777777778


np.float64(0.7777777777777778)

In [14]:
#WANTED TO DO FINAL SCORE AVERAGE
#Full functions for every single song 
def recommendFinal (track_name,k=10):
    index = get_indexFinal(track_name)
    if index is None:
        return 0
    recommended_songs, indices = get_reccsFinal(index)
    score = get_similarity_scoreFinal(index,indices)
    return score
def get_indexFinal(track_name):
    matches = df[(df["track"].str.lower() == track_name.lower())]
    if matches.empty:
        print("Track name not found. Try again.")
        return None
    return matches.index[0]

def get_reccsFinal(song_index):
    distances, indices = model.kneighbors(X_scaled[song_index].reshape(1, -1))
    indices = indices[0][1:]
    recommended_songs = df.iloc[indices]
    return recommended_songs, indices

#Overall overlap
def get_similarity_scoreFinal(song_index,indices):
    
    original = df.iloc[song_index][genre_cols]
    recs = df.iloc[indices][genre_cols]
    
    active_genres = original[original == 1].index
    if len(active_genres) == 0 or len(indices) == 0:
        return 0
    matches = recs[active_genres].sum(axis=1)
    
    consistency_score = matches.mean() / len(active_genres)
    return consistency_score

In [15]:
score = 0
size =0
for i in range(len(df)):
    name = df.iloc[i]["track"]
    scoreCurr = recommendFinal(name)
    score+=scoreCurr
    size+=1
print("Overall Average: ",score/size)



Overall Average:  0.8873200390529074
